# grid-rbd torch backend — autograd & a tiny IK

The torch backend returns `torch.Tensor`s from CUDA-resident kernels and is
**autograd-aware** for `rnea` / `forward_dynamics` / `aba` / `integrator`
(analytic backward via the `*_gradient` kernels). We show a gradient flowing
through `forward_dynamics`, an analytic-vs-finite-difference check (kernels are
float32-only, so we use FD rather than `torch.autograd.gradcheck`), and a tiny
end-effector IK by gradient descent.

Requires: CUDA GPU + `nvcc` + a CUDA-capable **torch** build (on sm_120 you
need a cu128+ wheel). iiwa14 = seconds to compile (cache-hit on re-run).

In [ ]:
import numpy as np, torch
from pathlib import Path
import grid_rbd
assert torch.cuda.is_available(), 'need a CUDA torch build'
URDF = Path.cwd().parent / 'robot_assets' / 'iiwa14.urdf'
torch.manual_seed(0); np.random.seed(0)
dev = 'cuda'

## 1. Register with the torch backend

In [ ]:
h = grid_rbd.register_robot('iiwa14_torch_nb', urdf_path=str(URDF),
                            floating_base=False, max_batch_size=64,
                            backend='torch')
print(h)
NJ = h.num_joints

## 2. A gradient flows through forward_dynamics

In [ ]:
B = 4
q  = torch.randn(B, NJ, device=dev, requires_grad=True)
qd = torch.randn(B, NJ, device=dev, requires_grad=True)
u  = torch.randn(B, NJ, device=dev, requires_grad=True)
qdd = h.forward_dynamics(q, qd, u)
loss = qdd.pow(2).sum()
loss.backward()
print('q.grad finite:', bool(torch.isfinite(q.grad).all()),
      ' u.grad finite:', bool(torch.isfinite(u.grad).all()))
assert q.grad is not None and torch.isfinite(q.grad).all()
assert u.grad is not None and torch.isfinite(u.grad).all()

## 3. Analytic backward vs finite-difference VJP (float32 tol)

We compare the custom backward's vector-Jacobian product against a
central-difference VJP of the forward op.

In [ ]:
torch.manual_seed(1)
q0  = torch.randn(1, NJ, device=dev)
qd0 = torch.randn(1, NJ, device=dev)
u0  = torch.randn(1, NJ, device=dev)
v   = torch.randn(1, NJ, device=dev)  # upstream cotangent

qa = q0.clone().requires_grad_(True)
out = h.forward_dynamics(qa, qd0, u0)
(out * v).sum().backward()
analytic = qa.grad.clone()

eps = 1e-2
fd = torch.zeros_like(q0)
for j in range(NJ):
    dq = torch.zeros_like(q0); dq[0, j] = eps
    fp = h.forward_dynamics(q0 + dq, qd0, u0)
    fm = h.forward_dynamics(q0 - dq, qd0, u0)
    fd[0, j] = ((fp - fm) / (2*eps) * v).sum()
rel = (analytic - fd).norm() / (fd.norm() + 1e-6)
print('analytic-vs-FD relative error:', float(rel))
assert rel < 5e-2, rel

## 4. Tiny IK: descend ‖ee_pose_xyz(q) − target‖²

`end_effector_pose` is forward-only, but `end_effector_pose_gradient` gives
the spatial Jacobian; we use it to take explicit Gauss-style steps on the
position error and check the loss decreases.

In [ ]:
q_ik = torch.randn(1, NJ, device=dev) * 0.3
target = h.end_effector_pose(torch.zeros(1, NJ, device=dev))[:, :3].clone()
losses = []
for it in range(40):
    pose = h.end_effector_pose(q_ik)
    err = pose[:, :3] - target            # (1,3) position error
    losses.append(float(err.pow(2).sum()))
    J = h.end_effector_pose_gradient(q_ik) # (1, 6, NV)
    Jp = J[:, 3:6, :]                      # linear (xyz) rows
    # gradient of 1/2||err||^2 wrt q  =  Jp^T err
    g = torch.bmm(Jp.transpose(1, 2), err.unsqueeze(-1)).squeeze(-1)
    q_ik = q_ik - 2.0 * g
print('IK loss: start %.4e -> end %.4e' % (losses[0], losses[-1]))
assert losses[-1] < losses[0]
assert losses[-1] < 1e-3